In [2]:
from scipy.io import arff
import pandas as pd
import numpy as np
from scipy.io import arff
from collections import Counter
from Class import *
from pathlib import Path
import joblib
from typing import Tuple, Union
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import accuracy_score
from sklearn.model_selection import KFold
from sklearn.base import clone
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestClassifier, BaggingClassifier, GradientBoostingClassifier
from sklearn.neural_network import MLPClassifier
from sklearn.svm import SVC
from sklearn.neighbors import KNeighborsClassifier
from sklearn.naive_bayes import GaussianNB
from sklearn.linear_model import LogisticRegression
from sklearn.utils import shuffle

In [5]:
def voting(labels, scores=None):
    labels = np.asarray(labels)
    final = []
        
    for j in range(labels.shape[1]):
        values, counts = np.unique(labels[:, j], return_counts=True)
        final.append(values[np.argmax(counts)])

    return np.array(final)
    
def top5_voting(labels, scores):
    labels = np.asarray(labels)
    final = []
    
    scores = np.asarray(scores)
    top_5_indices = np.argsort(scores)[-5:]

    for j in range(labels.shape[1]):
        top_5_predictions = labels[top_5_indices, j]
        values, counts = np.unique(top_5_predictions, return_counts=True)
        final.append(values[np.argmax(counts)])

    return np.array(final)
    
def weighted_voting(labels, scores):
    labels = np.asarray(labels)
    scores = np.asarray(scores)

    n_models, n_samples = labels.shape
    n_classes = labels.max() + 1

    score_matrix = np.zeros((n_samples, n_classes))

    for i in range(n_models):
        score_matrix[np.arange(n_samples), labels[i]] += scores[i]

    final_labels = np.argmax(score_matrix, axis=1)
    return final_labels

def top5_weighted_voting(labels, scores):
    labels = np.asarray(labels)
    scores = np.asarray(scores)

    n_models, n_samples = labels.shape
    
    top_5_indices = np.argsort(scores)[-5:]
    
    n_classes = labels.max() + 1
    score_matrix = np.zeros((n_samples, n_classes))

    for i in top_5_indices:
        score_matrix[np.arange(n_samples), labels[i]] += scores[i]

    final_labels = np.argmax(score_matrix, axis=1)
    return final_labels

In [6]:
def create_builtin_models(random_state=42):
    """
    Built-in classifiers available in scikit-learn.

    X must be numeric.
    y should be a one-dimensional vector of class labels.
    """
    return {
        "Random Forest": RandomForestClassifier(
            n_estimators=100,
            random_state=random_state,
            n_jobs=-1
        ),

        "MLP": make_pipeline(
            StandardScaler(),
            MLPClassifier(
                hidden_layer_sizes=(100,),
                max_iter=1000,
                random_state=random_state
            )
        ),

        "Bagging": BaggingClassifier(
            estimator=DecisionTreeClassifier(
                random_state=random_state
            ),
            n_estimators=100,
            random_state=random_state,
            n_jobs=-1
        ),

        "SVM": make_pipeline(
            StandardScaler(),
            SVC(
                kernel="rbf",
                C=1.0,
                random_state=random_state
            )
        ),

        "CART": DecisionTreeClassifier(
            criterion="gini",
            random_state=random_state
        ),

        "IBk": make_pipeline(
            StandardScaler(),
            KNeighborsClassifier(
                n_neighbors=5
            )
        ),

        "Naive Bayes": GaussianNB()
    }

In [7]:
def summarize_results(fold_accuracies):
    results = {}

    print(f"\n{'=' * 17} Final Results {'=' * 17}")

    for model_name, scores in fold_accuracies.items():
        scores = np.asarray(scores, dtype=np.float64)

        valid_scores = scores[~np.isnan(scores)]

        if len(valid_scores) == 0:
            mean_accuracy = np.nan
            std_accuracy = np.nan
        else:
            mean_accuracy = np.mean(valid_scores)
            std_accuracy = np.std(valid_scores)

        results[model_name] = {
            "mean_accuracy": float(mean_accuracy),
            "std_accuracy": float(std_accuracy),
            "fold_accuracies": scores.tolist()
        }

        if np.isnan(mean_accuracy):
            print(f"{model_name:<20}: failed")
        else:
            print(
                f"{model_name:<20}: "
                f"{mean_accuracy * 100:.2f}% "
                f"± {std_accuracy * 100:.2f}%"
            )

    return results

In [8]:
def extract_rules(model):
    return {
        "rules_if": np.asarray(
            model._final_rules_if
        ).copy(),

        "rules_then": np.asarray(
            model._final_rules_then
        ).copy(),

        "fitnesses": np.asarray(
            model._fitnesses
        ).copy(),

        "predicted_classes": np.argmax(
            model._final_rules_then,
            axis=1
        ),

        "number_of_rules": model.getNumOfRules(),

        "alpha": model._alpha,
        "gamma": model._gamma,
        "fitness_function": model._fitness_fn_name
    }

In [9]:
methods = {
    "voting": voting,
    "top5_voting": top5_voting,
    "weighted_voting": weighted_voting,
    "top5_weighted_voting": top5_weighted_voting
}

stackers = {
    'LogReg': LogisticRegression(random_state=42),
    'RF': RandomForestClassifier(n_estimators=50, max_depth=5, random_state=42),
    'GB': GradientBoostingClassifier(n_estimators=50, max_depth=3, random_state=42)
}

def train(
    X,
    y,
    X_Num,
    y_Num,
    kf,
    max_alpha,
    n_ensemble_models=30,
    random_state=42,
    save_path=None
):

    X = np.asarray(X, dtype=bool)
    y = np.asarray(y, dtype=bool)

    X_Num = np.asarray(X_Num, dtype=float)
    y_Num_1d = np.asarray(y_Num).ravel()

    if len(X) != len(X_Num):
        raise ValueError(
            "X and X_Num must contain the same number of samples."
        )

    if len(y) != len(y_Num_1d):
        raise ValueError(
            "y and y_Num must contain the same number of samples."
        )

    if np.any([
        value is None or (
            isinstance(value, float) and np.isnan(value)
        )
        for value in y_Num_1d
    ]):
        raise ValueError(
            "y_Num contains missing class labels. "
            "Remove those rows before cross-validation."
        )

    builtin_models = create_builtin_models(random_state)

    racer_configurations = {
        "RACER": None,
        "ROPAC_M": 1,
        "ROPAC_L": 2
    }

    result_names = (
        list(racer_configurations.keys())
        + list(methods.keys())
        + list(stackers.keys())
        + list(builtin_models.keys())
    )

    fold_accuracies = {
        name: []
        for name in result_names
    }

    splits = list(kf.split(X_Num, y_Num_1d))

    if save_path is not None:
        save_path = Path(save_path)
        save_path.mkdir(
            parents=True,
            exist_ok=True
        )

        joblib.dump(
            splits,
            save_path / "cv_splits.joblib"
        )

        joblib.dump(
            {
                "alpha": max_alpha,
                "gamma": 0.6,
                "n_ensemble_models": n_ensemble_models,
                "random_state": random_state,
                "kf": kf
            },
            save_path / "parameters.joblib"
        )


    seed_sequence = np.random.SeedSequence(random_state)
    fold_seeds = seed_sequence.spawn(len(splits))

    for fold_number, ((train_index, test_index), fold_seed) in enumerate(
        zip(splits, fold_seeds),
        start=1
    ):
        print(f"\n{'=' * 18} Fold {fold_number} {'=' * 18}")

        fold_rule_sets = {}
        ensemble_rule_sets = []
        fold_predictions = {}

        # -----------------------------------------------------
        # Boolean data: only for RACER and ROPAC
        # -----------------------------------------------------
        X_train = X[train_index]
        X_test = X[test_index]

        Y_train = y[train_index]
        Y_test = y[test_index]

        Y_train_eval = np.argmax(Y_train, axis=1)
        Y_test_eval = np.argmax(Y_test, axis=1)

        # -----------------------------------------------------
        # Numeric data: only for built-in classifiers
        # -----------------------------------------------------
        X_train_Num = X_Num[train_index]
        X_test_Num = X_Num[test_index]

        Y_train_Num = y_Num_1d[train_index]
        Y_test_Num = y_Num_1d[test_index]

        # =====================================================
        # RACER, ROPAC-M and ROPAC-L
        # =====================================================
        for model_name, ordering_method in racer_configurations.items():
            racer_model = ROPAC(
                alpha=max_alpha,
                gamma=0.6,
                suppress_warnings=True
            )
            if ordering_method is None:
                racer_model.fit(
                    X_train,
                    Y_train
                )
            else:
                racer_model.fit(
                    X_train,
                    Y_train,
                    ordering_method
                )
            racer_model.reduceRules()

            fold_rule_sets[model_name] = extract_rules(racer_model)
            prediction = np.asarray(
                racer_model.predict(X_test)
            ).ravel()
            fold_predictions[model_name] = prediction

            score = accuracy_score(Y_test_eval, prediction)

            fold_accuracies[model_name].append(float(score))

            print(
                f"{model_name:<20}: "
                f"{score * 100:.2f}%"
            )

        # =====================================================
        # MORCE ensemble
        # =====================================================
        rng = np.random.default_rng(fold_seed)

        train_labels = []
        test_labels = []
        training_scores = []

        for ensemble_index in range(n_ensemble_models):
            shuffled_indices = rng.permutation(
                len(X_train)
            )

            X_shuffled = X_train[shuffled_indices]
            Y_shuffled = Y_train[shuffled_indices]

            ropac_model = ROPAC(
                alpha=max_alpha,
                gamma=0.6,
                suppress_warnings=True
            )

            ropac_model.fit(X_shuffled, Y_shuffled)
            ropac_model.reduceRules()
            fold_rule_sets[model_name] = extract_rules(racer_model)

            train_prediction = ropac_model.predict(
                X_train
            )

            test_prediction = ropac_model.predict(
                X_test
            )

            train_labels.append(
                np.asarray(train_prediction).ravel()
            )

            test_labels.append(
                np.asarray(test_prediction).ravel()
            )

            training_score = ropac_model.score(
                X_train,
                Y_train
            )

            training_scores.append(
                float(training_score)
            )

            ensemble_rule_sets.append({
                "model_index": ensemble_index,
                "shuffled_indices": shuffled_indices.copy(),
                "training_score": float(training_score),
                "rules": extract_rules(ropac_model)
            })
        
        train_meta = np.asarray(train_labels).T
        test_meta = np.asarray(test_labels).T

        # =====================================================
        # MORCE aggregation functions
        # =====================================================
        for method_name, aggregation_function in methods.items():
            final_prediction = aggregation_function(
                test_labels,
                training_scores
            )
            fold_predictions[method_name] = np.asarray(
                final_prediction
            ).ravel()

            accuracy = accuracy_score(
                Y_test_eval,
                final_prediction
            )

            fold_accuracies[method_name].append(
                float(accuracy)
            )

            print(
                f"{method_name:<20}: "
                f"{accuracy * 100:.2f}%"
            )
        
        # =====================================================
        # MORCE stacking classifiers
        # =====================================================
        for stacker_name, stacker in stackers.items():
            stacker_model = clone(stacker)

            stacker_model.fit(
                train_meta,
                Y_train_eval
            )

            final_prediction = stacker_model.predict(
                test_meta
            )
            fold_predictions[stacker_name] = np.asarray(
                final_prediction
            ).ravel()

            accuracy = accuracy_score(
                Y_test_eval,
                final_prediction
            )

            fold_accuracies[stacker_name].append(
                float(accuracy)
            )

            print(
                f"{stacker_name:<20}: "
                f"{accuracy * 100:.2f}%"
            )
    
        # =====================================================
        # Built-in classifiers: numeric data only
        # =====================================================
        for model_name, estimator in builtin_models.items():
            model = clone(estimator)

            try:
                model.fit(
                    X_train_Num,
                    Y_train_Num
                )

                prediction = model.predict(
                    X_test_Num
                )
                fold_predictions[model_name] = np.asarray(
                    prediction
                ).ravel()

                accuracy = accuracy_score(
                    Y_test_Num,
                    prediction
                )

                fold_accuracies[model_name].append(
                    float(accuracy)
                )

                print(
                    f"{model_name:<20}: "
                    f"{accuracy * 100:.2f}%"
                )

            except Exception as error:
                fold_accuracies[model_name].append(
                    np.nan
                )

                print(
                    f"{model_name:<20}: failed — {error}"
                )

        if save_path is not None:
            fold_data = {
                "fold_number": fold_number,
                "train_indices": train_index.copy(),
                "test_indices": test_index.copy(),

                "y_train": Y_train_eval.copy(),
                "y_test": Y_test_eval.copy(),
                
                "y_train_num": Y_train_Num.copy(),
                "y_test_num": Y_test_Num.copy(),

                "base_train_predictions": np.asarray(
                    train_labels
                ),

                "base_test_predictions": np.asarray(
                    test_labels
                ),

                "training_scores": np.asarray(
                    training_scores
                ),

                "predictions": fold_predictions,

                "rule_sets": fold_rule_sets,
                "ensemble_rule_sets": ensemble_rule_sets,

                "alpha": max_alpha,
                "gamma": 0.6,
                "random_state": random_state
            }

            joblib.dump(
                fold_data,
                save_path / f"fold_{fold_number}.joblib",
                compress=3
            )
            
    return summarize_results(fold_accuracies)

# Data Cleaning

In [11]:
datasetname = "4-kidney"
data_file = "../data/{0}.arff".format(datasetname)

In [12]:
# csv
df = pd.read_csv(
        data_file,
        header= None,
        sep=",",
        na_values="?",
        skipinitialspace=True, 
        # encoding="utf-16"
    )

In [166]:
# xlsx
df = pd.read_excel(data_file, sheet_name=0)

In [13]:
df

,0,1,2,3,4,5,6
0,1,8,1,28,male,Other,N
1,1,16,1,28,male,Other,N
2,2,23,1,48,female,Glomerulo_Nephritis,N
3,2,13,0,48,female,Glomerulo_Nephritis,N
4,3,22,1,32,male,Other,N
...,...,...,...,...,...,...,...
71,36,16,0,42,female,Other,P
72,37,6,0,52,female,Polycystic_Kidney_Disease,N
73,37,78,1,52,female,Polycystic_Kidney_Disease,N
74,38,63,1,60,male,Polycystic_Kidney_Disease,N


In [ ]:
# for feature vector datasets
import re

def load_sparse_data(file_path):
    rows = []

    with open(file_path, "r") as f:
        for line in f:
            line = line.strip().strip("{}")

            pairs = line.split(",")

            row = {}

            for pair in pairs:
                idx, value = pair.strip().split()
                row[int(idx)] = float(value)

            rows.append(row)

    # find number of attributes
    n_features = max(
        max(row.keys()) for row in rows
    ) + 1

    # create dense matrix
    data = np.zeros((len(rows), n_features))

    for i, row in enumerate(rows):
        for col, value in row.items():
            data[i, col] = value

    df = pd.DataFrame(data)

    return data, df

data, df = load_sparse_data(data_file)

In [ ]:
# if target column needs swapping
cols = list(df.columns)
cols[1], cols[2] = cols[2], cols[1]
df = df[cols]

In [132]:
# remove first row if it contains column names
df = df.iloc[1:].reset_index(drop=True)

In [174]:
# remove first column if it contains ID
df = df.iloc[:, 1:].reset_index(drop=True)

In [15]:
dataTypes = []
for col in df.columns:
    if df[col].dropna().map(type).eq(bool).all():
        dataTypes.append("nominal")
    
    else:
        numeric = pd.to_numeric(df[col], errors="coerce")

        if numeric.notna().sum() == df[col].notna().sum():
            df[col] = numeric
            dataTypes.append("numeric")
        else:
            dataTypes.append("nominal")
dataTypes

['numeric', 'numeric', 'numeric', 'numeric', 'nominal', 'nominal', 'nominal']

In [16]:
# based on target column placement

classIndex=len(df.columns) - 1
# classIndex=0

In [17]:
from sklearn.impute import SimpleImputer

dataNum = df.copy()
dataSet = dataNum.to_numpy(dtype=object)

# Boolian Dataset
preprocessor = ROPACPreprocesser()
X, y = preprocessor.fit_transform(dataSet.copy(), dataTypes, classIndex)

X = np.asarray(X, dtype=bool)
y = np.asarray(y, dtype=bool)

# Numeric dataset
catAttrIndex = [
    i for i, t in enumerate(dataTypes)
    if t not in ["numeric", "integer", "real"]
    and i != classIndex
]
if catAttrIndex:
    for i in catAttrIndex:
        dataNum.iloc[:, i] = dataNum.iloc[:, i].astype("category").cat.codes

X_Num = dataNum.drop(
    columns=dataNum.columns[classIndex]
).to_numpy(dtype=float)

# Use exactly the same target encoding as ROPAC
y_Num = np.argmax(y, axis=1)

# Shuffle the data
rng = np.random.default_rng()
idx = rng.permutation(len(X))

X = X[idx]
y = y[idx]

X_Num = X_Num[idx]
y_Num = y_Num[idx]


imputer = SimpleImputer(strategy="mean")
X_Num = imputer.fit_transform(X_Num)

kf = KFold(n_splits=10, shuffle=False)

# Finding best Alpha

In [18]:
racer = ROPAC(gamma=0.6, suppress_warnings=True)
max=0
min=1
while True:
    score = 0
    for train_index, test_index in kf.split(X):
        X_train, X_test = X[train_index], X[test_index]
        Y_train, Y_test = y[train_index], y[test_index]

        racer.fit(X_train, Y_train)
        racer.reduceRules()
        score += racer.score(X_test, Y_test)

    if float(score/10) > max :
        max = float(score/10)
        maxalpha = racer._alpha

    if float(score/10) < min :
        min = float(score/10)
        minalpha = racer._alpha
    
    racer._alpha -= 0.01
    racer._beta = 1 - racer._alpha
    if racer._alpha<0.5:
        break

max, maxalpha, min, minalpha

Labels for these instances will be determined by a closest match algorithm.
Labels for these instances will be determined by a closest match algorithm.
Labels for these instances will be determined by a closest match algorithm.
Labels for these instances will be determined by a closest match algorithm.
Labels for these instances will be determined by a closest match algorithm.
Labels for these instances will be determined by a closest match algorithm.
Labels for these instances will be determined by a closest match algorithm.
Labels for these instances will be determined by a closest match algorithm.
Labels for these instances will be determined by a closest match algorithm.
Labels for these instances will be determined by a closest match algorithm.
Labels for these instances will be determined by a closest match algorithm.
Labels for these instances will be determined by a closest match algorithm.
Labels for these instances will be determined by a closest match algorithm.
Labels for t

(0.775, 0.99, 0.5125, 0.6499999999999997)

In [ ]:
# for large datasets
maxalpha = 0.99
minalpha = 0.50

# Train

In [19]:
results = train(
    X=X,
    y=y,
    X_Num=X_Num,
    y_Num=y_Num,
    kf=kf,
    max_alpha=maxalpha,
    n_ensemble_models=30,
    random_state=42,
    save_path=f"../saved_results/{datasetname}"
)


================== Fold 1 ==================
RACER               : 75.00%
ROPAC_M             : 75.00%
ROPAC_L             : 75.00%
voting              : 75.00%
top5_voting         : 75.00%
weighted_voting     : 75.00%
top5_weighted_voting: 75.00%
LogReg              : 75.00%
RF                  : 75.00%
GB                  : 75.00%
Random Forest       : 100.00%


c:\Users\WIN 10\anaconda3\Lib\site-packages\sklearn\neural_network\_multilayer_perceptron.py:781: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (1000) reached and the optimization hasn't converged yet.
  warnings.warn(


MLP                 : 87.50%
Bagging             : 100.00%
SVM                 : 87.50%
CART                : 87.50%
IBk                 : 100.00%
Naive Bayes         : 75.00%

================== Fold 2 ==================
RACER               : 87.50%
ROPAC_M             : 75.00%
ROPAC_L             : 87.50%
voting              : 87.50%
top5_voting         : 87.50%
weighted_voting     : 87.50%
top5_weighted_voting: 87.50%
LogReg              : 87.50%
RF                  : 87.50%
GB                  : 87.50%
Random Forest       : 100.00%
MLP                 : 75.00%
Bagging             : 75.00%
SVM                 : 75.00%
CART                : 75.00%
IBk                 : 50.00%
Naive Bayes         : 62.50%

================== Fold 3 ==================
RACER               : 50.00%
ROPAC_M             : 50.00%
ROPAC_L             : 50.00%
voting              : 50.00%
top5_voting         : 50.00%
weighted_voting     : 50.00%
top5_weighted_voting: 50.00%
LogReg              : 50.00%
RF    

c:\Users\WIN 10\anaconda3\Lib\site-packages\sklearn\neural_network\_multilayer_perceptron.py:781: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (1000) reached and the optimization hasn't converged yet.
  warnings.warn(


MLP                 : 87.50%
Bagging             : 75.00%
SVM                 : 62.50%
CART                : 87.50%
IBk                 : 50.00%
Naive Bayes         : 100.00%

================== Fold 6 ==================
RACER               : 75.00%
Labels for these instances will be determined by a closest match algorithm.
ROPAC_M             : 62.50%
ROPAC_L             : 75.00%
voting              : 75.00%
top5_voting         : 75.00%
weighted_voting     : 75.00%
top5_weighted_voting: 75.00%
LogReg              : 75.00%
RF                  : 75.00%
GB                  : 75.00%
Random Forest       : 62.50%
MLP                 : 62.50%
Bagging             : 62.50%
SVM                 : 50.00%
CART                : 75.00%
IBk                 : 50.00%
Naive Bayes         : 37.50%

================== Fold 7 ==================
RACER               : 71.43%
ROPAC_M             : 71.43%
ROPAC_L             : 71.43%
voting              : 71.43%
top5_voting         : 71.43%
weighted_voting    

c:\Users\WIN 10\anaconda3\Lib\site-packages\sklearn\neural_network\_multilayer_perceptron.py:781: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (1000) reached and the optimization hasn't converged yet.
  warnings.warn(


MLP                 : 100.00%
Bagging             : 100.00%
SVM                 : 85.71%
CART                : 57.14%
IBk                 : 100.00%
Naive Bayes         : 57.14%

================== Fold 10 ==================
RACER               : 71.43%
ROPAC_M             : 71.43%
ROPAC_L             : 71.43%
voting              : 71.43%
top5_voting         : 71.43%
weighted_voting     : 71.43%
top5_weighted_voting: 71.43%
LogReg              : 71.43%
RF                  : 71.43%
GB                  : 71.43%
Random Forest       : 85.71%
MLP                 : 71.43%
Bagging             : 85.71%
SVM                 : 71.43%
CART                : 85.71%
IBk                 : 71.43%
Naive Bayes         : 71.43%

================= Final Results =================
RACER               : 77.50% ± 15.71%
ROPAC_M             : 75.00% ± 15.91%
ROPAC_L             : 78.93% ± 14.39%
voting              : 77.50% ± 15.71%
top5_voting         : 77.50% ± 15.71%
weighted_voting     : 77.50% ± 15.71%
top5

In [20]:
from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    matthews_corrcoef
)

BUILTIN_MODELS = {
    "Random Forest",
    "MLP",
    "Bagging",
    "SVM",
    "CART",
    "IBk",
    "Naive Bayes"
}

def evaluate_saved_algorithms(
    save_path,
    average="weighted"
):
    save_path = Path(save_path)

    fold_files = sorted(
        save_path.glob("fold_*.joblib")
    )

    if not fold_files:
        raise FileNotFoundError(
            f"No fold files were found in: {save_path}"
        )

    metric_rows = []

    for fold_file in fold_files:
        fold_data = joblib.load(fold_file)
        fold_number = fold_data["fold_number"]
        saved_predictions = fold_data[
            "predictions"
        ]

        for algorithm_name, y_pred in saved_predictions.items():
            
            if algorithm_name in BUILTIN_MODELS:
                y_true = np.asarray(
                    fold_data["y_test_num"]
                ).ravel()
            else:
                y_true = np.asarray(
                    fold_data["y_test"]
                ).ravel()

            y_pred = np.asarray(y_pred).ravel()
            
            if y_pred is None:
                continue

            if len(y_true) != len(y_pred):
                print(
                    f"Skipping {algorithm_name} in fold "
                    f"{fold_number}: prediction length mismatch."
                )
                continue

            metric_rows.append({
                "Fold": fold_number,
                "Algorithm": algorithm_name,

                "Accuracy": accuracy_score(
                    y_true,
                    y_pred
                ),

                "Balanced Accuracy": balanced_accuracy_score(
                    y_true,
                    y_pred
                ),

                "Precision": precision_score(
                    y_true,
                    y_pred,
                    average=average,
                    zero_division=0
                ),

                "Recall": recall_score(
                    y_true,
                    y_pred,
                    average=average,
                    zero_division=0
                ),

                "F1": f1_score(
                    y_true,
                    y_pred,
                    average=average,
                    zero_division=0
                ),

                "MCC": matthews_corrcoef(
                    y_true,
                    y_pred
                )
            })

    fold_results = pd.DataFrame(metric_rows)

    if fold_results.empty:
        raise ValueError(
            "No valid saved predictions were found."
        )

    summary_results = (
        fold_results
        .groupby("Algorithm", sort=False)
        .agg({
            "Accuracy": ["mean", "std"],
            "Balanced Accuracy": ["mean", "std"],
            "Precision": ["mean", "std"],
            "Recall": ["mean", "std"],
            "F1": ["mean", "std"],
            "MCC": ["mean", "std"]
        })
        .reset_index()
    )

    summary_results.columns = [
        "Algorithm",
        "Accuracy Mean",
        "Accuracy Std",
        "Balanced Accuracy Mean",
        "Balanced Accuracy Std",
        "Precision Mean",
        "Precision Std",
        "Recall Mean",
        "Recall Std",
        "F1 Mean",
        "F1 Std",
        "MCC Mean",
        "MCC Std"
    ]

    return fold_results, summary_results

In [21]:
fold_metrics, summary_metrics = evaluate_saved_algorithms(
    save_path=f"../saved_results/{datasetname}",
    average="weighted"
)

fold_metrics_t = fold_metrics.set_index(fold_metrics.columns[0]).T
summary_metrics_t = summary_metrics.set_index(summary_metrics.columns[0]).T

fold_metrics_t.to_excel(
    f"../saved_results/{datasetname}/fold_metrics.xlsx"
)

summary_metrics_t.to_excel(
    f"../saved_results/{datasetname}/summary_metrics.xlsx"
)

summary_metrics_t

Algorithm,RACER,ROPAC_M,ROPAC_L,voting,top5_voting,weighted_voting,top5_weighted_voting,LogReg,RF,GB,Random Forest,MLP,Bagging,SVM,CART,IBk,Naive Bayes
Accuracy Mean,0.775000,0.750000,0.789286,0.775000,0.775000,0.775000,0.775000,0.775000,0.775000,0.775000,0.869643,0.750000,0.819643,0.673214,0.789286,0.685714,0.682143
Accuracy Std,0.165643,0.167726,0.151710,0.165643,0.165643,0.165643,0.165643,0.165643,0.165643,0.165643,0.144595,0.157035,0.132419,0.156820,0.095535,0.193283,0.162969
Balanced Accuracy Mean,0.769167,0.744167,0.781667,0.769167,0.769167,0.769167,0.769167,0.769167,0.769167,0.769167,0.862500,0.748333,0.819167,0.660833,0.796667,0.680000,0.673333
Balanced Accuracy Std,0.167270,0.168373,0.156140,0.167270,0.167270,0.167270,0.167270,0.167270,0.167270,0.167270,0.145734,0.155248,0.134374,0.155011,0.099551,0.202233,0.172527
Precision Mean,0.791905,0.765238,0.803810,0.791905,0.791905,0.791905,0.791905,0.791905,0.791905,0.791905,0.897917,0.769256,0.849048,0.701905,0.832917,0.702619,0.691224
Precision Std,0.164557,0.166686,0.152621,0.164557,0.164557,0.164557,0.164557,0.164557,0.164557,0.164557,0.117665,0.162115,0.117150,0.184330,0.096664,0.204819,0.212678
Recall Mean,0.775000,0.750000,0.789286,0.775000,0.775000,0.775000,0.775000,0.775000,0.775000,0.775000,0.869643,0.750000,0.819643,0.673214,0.789286,0.685714,0.682143
Recall Std,0.165643,0.167726,0.151710,0.165643,0.165643,0.165643,0.165643,0.165643,0.165643,0.165643,0.144595,0.157035,0.132419,0.156820,0.095535,0.193283,0.162969
F1 Mean,0.766984,0.741587,0.784603,0.766984,0.766984,0.766984,0.766984,0.766984,0.766984,0.766984,0.860960,0.743723,0.812078,0.662431,0.784048,0.672381,0.654668
F1 Std,0.173844,0.175179,0.154386,0.173844,0.173844,0.173844,0.173844,0.173844,0.173844,0.173844,0.157947,0.163120,0.143989,0.157204,0.098035,0.202336,0.185359


In [22]:
with pd.ExcelWriter(
    f"../saved_results/{datasetname}/evaluation_metrics.xlsx"
) as writer:
    fold_metrics.to_excel(
        writer,
        sheet_name="Fold Metrics",
        index=False
    )

    summary_metrics.to_excel(
        writer,
        sheet_name="Summary",
        index=False
    )

In [23]:
save_path = Path(f"../saved_results/{datasetname}")

parameters = joblib.load(
    save_path / "parameters.joblib"
)
parameters

{'alpha': 0.99,
 'gamma': 0.6,
 'n_ensemble_models': 30,
 'random_state': 42,
 'kf': KFold(n_splits=10, random_state=None, shuffle=False)}